# RAG Evaluation Test Set Generation

This example shows how to use the [Ragas](https://docs.ragas.io/en/stable/) (```v 0.1.22```) framework to generate a **test set** that can be used to evaluate the quality of a RAG pipeline. We then use the Python [LangChain](https://python.langchain.com/docs/introduction/) library to run some requests through this pipeline and we evaluate the quality of the results.

### <u>Requirements</u>
1. As you will accessing the LLMs and embedding models through Vector AI Engineering's Kaleidoscope Service (Vector Inference + Autoscaling), you will need to request a KScope API Key:

      Run the following command (replace ```<user_id>``` and ```<password>```) from **within the cluster** to obtain the API Key. The ```access_token``` in the output is your KScope API Key.
  ```bash
  curl -X POST -d "grant_type=password" -d "username=<user_id>" -d "password=<password>" https://kscope.vectorinstitute.ai/token
  ```
2. After obtaining the `.env` configurations, make sure to create the ```.kscope.env``` file in your home directory (```/h/<user_id>```) and set the following env variables:
- For local models through Kaleidoscope (KScope):
    ```bash
    export OPENAI_BASE_URL="https://kscope.vectorinstitute.ai/v1"
    export OPENAI_API_KEY=<kscope_api_key>
    ```
- For OpenAI models:
   ```bash
   export OPENAI_BASE_URL="https://api.openai.com/v1"
   export OPENAI_API_KEY=<openai_api_key>
   ```

## Set up the RAG workflow environment

#### Import libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
#!pip install pdfplumber


In [3]:
#import pdfplumber

In [4]:
import numpy as np
import os
import sys

from datasets import Dataset
from pathlib import Path

from langchain.chains import RetrievalQA
from langchain.document_loaders.pdf import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas import evaluate
from ragas.metrics import Faithfulness, ContextPrecision, AnswerCorrectness
from ragas.testset import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context

In [5]:
cd ..

/fs01/home/ws_ikharchuk/rag_bootcamp_ik/rag_evaluation


#### Load config files

In [6]:
# Add root folder of the rag_bootcamp repo to PYTHONPATH
current_dir = Path().resolve()
parent_dir = current_dir.parent
sys.path.insert(0, str(parent_dir))



In [7]:
from utils.load_secrets import load_env_file
load_env_file()

#### Set up some helper functions

In [8]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

#### Make sure other necessary items are in place

## Generate a sythentic test set

#### Start by loading in the documents we'll be using to augment our RAG generations

In [9]:
#load all documents

In [10]:

from langchain.document_loaders import TextLoader
from langchain.document_loaders import PyPDFLoader

In [11]:
%%time
directory_path = "/projects/RAG2/scotia-2/Datasets-Scotia-2/IBIS"
file_list = [
   '48412CA Long-Distance Freight Trucking in Canada Industry Report.pdf',
#'48422CA Local Specialized Freight Trucking in Canada Industry Report.pdf'
]  # Replace with your actual file names

# Load only the specified files
docs = []
for file_name in file_list:
    file_path = os.path.join (directory_path, file_name)
    loader = PyPDFLoader(file_path)
    docs.extend(loader.load())  # Append loaded pages to the list
print(f"Number of source documents: {len(docs)}")
for document in docs:
    document.metadata['file_name'] = document.metadata['source']

Number of source documents: 39
CPU times: user 1.45 s, sys: 58.3 ms, total: 1.5 s
Wall time: 1.53 s


In [12]:
%%time
# no need 
#Process PDFs
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=32)
chunks = text_splitter.split_documents(docs)
print(f"Number of text chunks: {len(chunks)}")

Number of text chunks: 47
CPU times: user 7.81 ms, sys: 0 ns, total: 7.81 ms
Wall time: 7.67 ms


#### Now use OpenAI to generate a test set from the data in these documents (This takes about 2-3 minutes)

**IMP Note:** The LLM and embedding model used for test set generation should be more capable than the model being evaluated. Hence, we will use OpenAI GPT-4o and OpenAI embeddings for this purpose.

Store your OpenAI API key in ```~/.ragas_openai.env``` using the following format (this is in addition to ```~/.kscope.env```):

```bash
export RAGAS_OPENAI_BASE_URL="https://api.openai.com/v1"
export RAGAS_OPENAI_API_KEY=<openai_api_key>
```

In [13]:
# Select  LLM (Llama)

In [14]:
from utils.load_secrets import load_env_file_ragas
load_env_file_ragas()

In [15]:
GENERATOR_BASE_URL = os.environ.get("OPENAI_BASE_URL")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

In [16]:
GENERATOR_MODEL_NAME = "Meta-Llama-3.1-8B-Instruct"
#GENERATOR_MODEL_NAME = 'DeepSeek-R1-Distill-Qwen-1.5B'
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

In [17]:
# llm = ChatOpenAI(
#     model=GENERATOR_MODEL_NAME,
#     temperature=0,
#     max_tokens=None,
#     base_url=GENERATOR_BASE_URL,
#     api_key=OPENAI_API_KEY
# )

In [18]:
generator_llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=os.environ["RAGAS_OPENAI_BASE_URL"],
    api_key=os.environ["RAGAS_OPENAI_API_KEY"],
)
generator_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url=os.environ["RAGAS_OPENAI_BASE_URL"],
    api_key=os.environ["RAGAS_OPENAI_API_KEY"],
)

In [19]:
# model_kwargs = {'device': 'cuda', 'trust_remote_code': True}
# encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

# print(f"Setting up the embeddings model...")
# embeddings = HuggingFaceEmbeddings(
#     model_name=   EMBEDDING_MODEL_NAME,
#     model_kwargs=model_kwargs,
#     encode_kwargs=encode_kwargs,
# )

In [20]:
def read_csv_from_directory(directory_path):
    dataframes= []
    for filename in os.listdir(directory_path):
        if filename.endswith('.csv'): 
            file_path = os.path.join(directory_path, filename)
            df = pd.read_csv(file_path)
            df["source"] = filename
            # print(df.head(1))
            dataframes.append(df)
    return pd.concat(dataframes, ignore_index=True)

In [21]:
#Load TRUCKING  data
file_paths = [#'/projects/RAG2/scotia-2/Datasets-Scotia-2/Agriculture_txt/agri_ca_co.csv', 
              '/projects/RAG2/scotia-2/Datasets-Scotia-2/Transport_txt/transport_CA.csv', 
              #'/projects/RAG2/scotia-2/Datasets-Scotia-2/Auto_txt/auto_ca.csv', 
             ]
def load_txt_file(file_path):
    # Create a TextLoader instance

    loader = TextLoader(file_path)

    # Load the document

    document = loader.load()
    chunks2 =text_splitter.split_documents(document)
    print(f"Number of text chunks: {len(chunks2)}")
    return chunks2, document

In [22]:
%%time
chunks =[]
for file_path in file_paths:
    chunks2, document_news = load_txt_file(file_path)
    chunks= chunks +chunks2

Number of text chunks: 403
CPU times: user 32 ms, sys: 4.55 ms, total: 36.6 ms
Wall time: 33.2 ms


In [23]:
len(document_news), len (docs)

(1, 39)

In [24]:
combined_docs = docs +document_news

In [41]:
len (combined_docs)

40

In [25]:
%%time
#generator_llm = (
#    model="DeepSeek-R1-Distill-Llama-8B",
#    base_url=os.environ["OPENAI_BASE_URL"],
#    api_key=os.environ["OPENAI_API_KEY"],
#)
# Define the RAG embeddings model (different than the OpenAI embedding model defined above for test set generation)
model_kwargs = {'device': 'cuda', 'trust_remote_code': True}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
# print(f"Setting up the RAG LLM...")
llm = ChatOpenAI(
    #model="DeepSeek-R1-Distill-Llama-8B",
    model="Meta-Llama-3.1-8B-Instruct",
    temperature=0,
    max_tokens=256,
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

CPU times: user 4.61 s, sys: 1.46 s, total: 6.07 s
Wall time: 6.89 s


In [67]:
#generator_llm.generate('what is  the day today?, Answer in no more then 10 words')

In [68]:
# %%time
# # Create generator with OpenAI model
# generator = TestsetGenerator.from_langchain(
#     generator_llm=generator_llm,
#     critic_llm=generator_llm,
#     embeddings=generator_embeddings,
# )

# # Generate the test set
# testset = generator.generate_with_langchain_docs(
#     documents=documents, 
#     test_size=1,
#     distributions={simple: 0.5, reasoning: 0.25, multi_context: 0.25},
# )

In [31]:
%%time
# Create generator with LLama model
generator = TestsetGenerator.from_langchain(
    #generator_llm=llm,
    generator_llm=generator_llm,
    #critic_llm=llm,
    critic_llm=generator_llm,
    #embeddings=embeddings,
    embeddings = generator_embeddings
)

# Generate the test set
testset = generator.generate_with_langchain_docs(
    documents=combined_docs, 
    test_size=50,
    distributions={simple: 0.5, reasoning: 0.25, multi_context: 0.25},
)

embedding nodes:   0%|          | 0/746 [00:00<?, ?it/s]

Filename and doc_id are the same for all nodes.


Generating:   0%|          | 0/50 [00:00<?, ?it/s]

CPU times: user 37.9 s, sys: 1.01 s, total: 39 s
Wall time: 5min 6s


#### Save dataset

In [32]:
testset2 = testset.to_pandas()

In [37]:
from collections import Counter

In [40]:
len (testset2.metadata[0])

1

In [33]:
testset2.tail(5)

,question,contexts,ground_truth,evolution_type,metadata,episode_done
45,What acquisitions did TFI make to boost its LT...,"[3.7%\n96.3%\n3.7 1,149.4 167.8 14.6 Compa...",TFI International made two significant acquisi...,multi_context,[{'source': '/projects/RAG2/scotia-2/Datasets-...,True
46,What challenges does the Mayan Train project h...,[ by the next five years\nas trade finance oft...,Experts have warned that the Mayan Train has r...,multi_context,[{'source': '/projects/RAG2/scotia-2/Datasets-...,True
47,What benefits do businesses see from integrate...,[Previous suppl y chain chal lenges ar e pr om...,Businesses see benefits from integrated supply...,multi_context,[{'source': '/projects/RAG2/scotia-2/Datasets-...,True
48,What steps did the Bombers take for Indigenous...,[ people from across\nCanada to attend the Win...,"The Bombers, in line with EIC's model, partici...",multi_context,[{'source': '/projects/RAG2/scotia-2/Datasets-...,True
49,What prompted Mayor Salinas's emergency declar...,[ late 2021 to help fund its\napproximately $3...,Mayor Salinas issued an emergency declaration ...,multi_context,[{'source': '/projects/RAG2/scotia-2/Datasets-...,True


In [42]:
import joblib

In [44]:


# Save (serialize) the object to a file
joblib.dump(testset, "../../Testing_Data/testset_combined.pkl")


['../../Testing_Data/testset_combined.pkl']

In [45]:
testset2.to_parquet ('../../Testing_Data/testset_combined.parquet')
testset2.to_csv ('../../Testing_Data/testset_combined.csv', index =False)

In [28]:
#load 
# Load (deserialize) the object from file
testset = joblib.load("../../Testing_Data/testset_combined.pkl")


## Now, start the RAG pipeline!

#### Choose the RAG LLM and embedding model
Note: This is different than the OpenAI LLM and embedding model defined above for test set generation.

In [46]:
RAG_LLM_MODEL_NAME = "Meta-Llama-3.1-8B-Instruct"
RAG_EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

#### Generate answers for all the questions in our test set

Go through the embedding, storage and retrieval steps.

In [47]:
# Split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=32)  ### Change Chunks here 
chunks = text_splitter.split_documents(combined_docs)
print(f"Number of text chunks: {len(chunks)}")

Number of text chunks: 450


In [48]:
# Define the RAG embeddings model (different than the OpenAI embedding model defined above for test set generation)
model_kwargs = {'device': 'cuda', 'trust_remote_code': True}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

print(f"Setting up the RAG embeddings model...")
embeddings = HuggingFaceEmbeddings(
    model_name=RAG_EMBEDDING_MODEL_NAME,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)
print (f"setting up {RAG_EMBEDDING_MODEL_NAME}")

Setting up the RAG embeddings model...
setting up BAAI/bge-base-en-v1.5


In [49]:
%%time
# Create the vector store and the retriever
vectorstore = FAISS.from_documents(chunks, #embeddings
                                  generator_embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [52]:
RAG_LLM_MODEL_NAME

'Meta-Llama-3.1-8B-Instruct'

In [38]:
%%time
# Define the RAG LLM (different than the OpenAI LLM defined above for test set generation)
print(f"Setting up the RAG LLM...")
llm = ChatOpenAI(
    model=RAG_LLM_MODEL_NAME,
    temperature=0,
    max_tokens=256,
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)
print (f"setting up {RAG_LLM_MODEL_NAME}")

Setting up the RAG LLM...
setting up Meta-Llama-3.1-8B-Instruct
CPU times: user 21 ms, sys: 4.14 ms, total: 25.1 ms
Wall time: 23.8 ms


Iterate over the questions in our synthetic testset, and run them each through the RAG pipeline to see what answers get returned. (This also takes 2-3 minutes)

In [53]:
# answer Questions

In [54]:
%%time
#USe OpenAi

dataset = testset.to_dataset()
answers = np.empty(len(dataset), dtype=object)

for index, row in enumerate(dataset):
    query = row["question"]
    
    # Run the query through the RAG pipeline
    rag_pipeline = RetrievalQA.from_llm(
        llm=generator_llm,
        retriever=retriever
    )
    answer = rag_pipeline.invoke(input=query)
    answer = answer["result"]
    print(f"Result {index}\nQuestion: {query}\nAnswer: {answer}\n")
    
    # Store the result
    answers[index] = answer

Result 0
Question: What percentage of structural damage is being reported in Jasper due to the wildfires?
Answer: Authorities have estimated that there is potentially 30% to 50% structural damage in Jasper due to the wildfires.

Result 1
Question: What is the expected growth range for adjusted diluted earnings per share according to CN's latest forecast?
Answer: I don't know.

Result 2
Question: What is the expected revenue growth for long-distance freight trucking services through the end of 2024?
Answer: The expected revenue growth for long-distance freight trucking services through the end of 2024 is a compound annual growth rate (CAGR) of 1.0%, reaching $31.3 billion.

Result 3
Question: What are the potential economic impacts of a labor stoppage at Canada's two largest railroad operators?
Answer: A labor stoppage at Canada's two largest railroad operators, Canadian National Railway and Canadian Pacific Kansas City, could have significant economic impacts. According to estimates, t

Result 13
Question: What accounting principles does CPKC follow for preparing financial information?
Answer: CPKC prepares financial information in accordance with accounting principles generally accepted in the United States of America (U.S. GAAP), unless otherwise noted.

Result 14
Question: What decision is the U.S. Federal Reserve expected to make regarding interest rates?
Answer: The U.S. Federal Reserve is expected to leave interest rates unchanged during its upcoming monetary policy decision.

Result 15
Question: What are the environmental benefits associated with the rail network's operations?
Answer: The environmental benefits associated with the rail network's operations include:

1. **Reduced Emissions**: CN has reduced locomotive emissions intensity by 45% since 1993, which contributes to lower greenhouse gas emissions.

2. **Fuel Efficiency**: The rail network consumes approximately 15% less fuel per gross ton mile than the industry average, indicating a more efficient use

Result 24
Question: What role do technology improvements play in enhancing the safety and efficiency of freight trucking?
Answer: Technology improvements play a significant role in enhancing the safety and efficiency of freight trucking in several ways:

1. **Safety Measures**: Technology has introduced various safety features aimed at ensuring that vehicles maintain safe operation. For example, advancements help prevent vehicles from veering off the road, accelerating too quickly, or traveling down steep grades at excessive speeds. 

2. **Data Collection and Analytics**: Companies are increasingly using technology to collect data on driving patterns, which can help identify risky behaviors and prevent potential accidents. This data-driven approach allows for better monitoring of drivers' day-to-day activities, leading to safer operations.

3. **Vehicle Upgrades**: The adoption of newer technologies, including sensors and advanced driving assistance systems, enhances vehicle performanc

Result 34
Question: What might the FedEx freight sale mean?
Answer: The potential sale or spin-off of FedEx's freight trucking business could have several implications:

1. **Increased Valuation**: Analysts have valued the FedEx Freight business at around $30 billion. If it is sold or operates as a standalone company, it may lead to a significant re-rating of its valuation, potentially increasing the overall market value for FedEx.

2. **Focus on Core Operations**: By divesting the freight business, FedEx could concentrate on its more profitable segments and streamline operations, which may enhance profitability.

3. **Market Dynamics**: The sale could impact competition within the logistics and freight industry, especially in light of Yellow Corp's recent operational shutdown, which has led to expectations of higher rates in the trucking sector.

4. **Investor Confidence**: FedEx's bullish annual profit forecast and the review of its Freight business may reassure investors about the c

Result 43
Question: What impacts do forward-looking statements in TFI's outlook have, and what external factors affect their reliability?
Answer: Forward-looking statements in TFI's outlook reflect management's current expectations regarding future results, performance, and achievements. However, these statements are inherently subject to significant business, economic, and competitive uncertainties and contingencies. 

External factors that affect the reliability of these forward-looking statements include:

1. Economic and geopolitical conditions.
2. Competition in the market.
3. Access to capital and market trends.
4. Changes in government regulations and funding, especially for sectors like health care.
5. Operational performance and growth metrics.
6. Environmental, social, and governance factors.
7. Risks associated with acquisitions and reliance on key customers.
8. Fluctuations in sales prices and purchase prices of assets.
9. Interest rates and foreign exchange rates.
10. Gene

In [36]:
%%time
#Use LLama
dataset = testset.to_dataset()
answers = np.empty(len(dataset), dtype=object)

for index, row in enumerate(dataset):
    query = row["question"]
    
    # Run the query through the RAG pipeline
    rag_pipeline = RetrievalQA.from_llm(
        llm=llm,
        retriever=retriever
    )
    answer = rag_pipeline.invoke(input=query)
    answer = answer["result"]
    print(f"Result {index}\nQuestion: {query}\nAnswer: {answer}\n")
    
    # Store the result
    answers[index] = answer

Result 0
Question: Here is a question that can be fully answered from the given context:

"What drives the demand for dry bulk transportation in the local specialized freight trucking industry in Canada?"

This question can be answered by referencing the context, which states that "Dry bulk transportation swells alongside construction activity" and that "Strong construction activity has supported this segment but is expected to trend downward through the end of 2024 as tightened monetary policy pressures businesses' financing ability.
Answer: According to the context, the demand for dry bulk transportation in the local specialized freight trucking industry in Canada is driven by construction activity. Specifically, it is stated that "Dry bulk transport is the largest market for specialized trucks" and that "A strong expansion in mining and high demand from the construction sector fueled market growth."

Result 1
Question: Here is a question that can be fully answered from the given con

Result 11
Question: Here is a question that can be fully answered from the given context:

"What are the expected trends in the Canadian long-distance specialized freight trucking industry, including the impact of technological advances and government policies?"

This question is fully answerable from the context, which discusses the industry's expected growth, the impact of technological advances, the effects of government policies such as the Express Entry initiative, and the industry's maturity and saturation levels.
Answer: Based on the provided context, here are the expected trends in the Canadian long-distance specialized freight trucking industry:

1. **Growth**: The industry is expected to grow, driven by the acceleration in the GDP growth from 1.1% in 2024 to 1.6% in 2025, and the expected CAGR of 1.9% in consumer spending through 2029. This growth will be fueled by the recovery in economic activity, manufacturing output, and consumer spending.
2. **Technological advances**: T

Result 21
Question: Here is a question that can be fully answered from the given context using the keyphrase "Barriers to Entry":

"What challenges do potential industry entrants face when entering the local specialized freight trucking industry in Canada?"

This question is fully answerable from the context, as it is directly addressed in the section "What challenges do potential industry entrants face?" under the subheading "Barriers to Entry".
Answer: According to the context, potential industry entrants face the following challenges when entering the local specialized freight trucking industry in Canada:

* Legal: The regulatory landscape is more demanding than for transporting regular freight, with regulations governing road safety, hours of service, weight, and the distribution of hazardous materials.
* Start-up Costs: Specialization type dictates the start-up costs incurred, with businesses targeting a specific niche requiring a truck compatible with the distribution service req

Result 32
Question: Here is the rewritten question:

"What could undermine the benefits of the company's expansion?"

I've kept the essence of the original question, but made it more indirect and concise by:

* Removing unnecessary words and phrases
* Using abbreviations (e.g. "the company's" becomes "the company")
* Changing the wording to make it more concise and natural-sounding.
Answer: Based on the provided context, I don't have enough information to determine what could undermine the benefits of the company's expansion. The context only provides information about various industries in Canada, including Wheat Farming, New Car Dealers, Auto Parts Manufacturing, and Long-Distance Specialized Freight Trucking. It does not mention a specific company or its expansion plans.

Result 33
Question: Here is a rewritten version of the question:

"What's the net change in Auto Parts Manufacturing businesses in Canada from 2024 to 2029?"

I've kept the essence of the original question, but mad

Result 42
Question: Here's a rewritten version of the question:

"What market conditions favor stable market share concentration for family-owned farms in Canada's corn industry?"

I've kept the essence of the original question, but made it more indirect and concise by:

* Removing unnecessary words and phrases
* Using abbreviations (e.g. "market conditions" instead of "market conditions do")
* Focusing on the key aspect of the question (stable market share concentration)
Answer: Based on the provided context, the market conditions that favor stable market share concentration for family-owned farms in Canada's corn industry are:

* Low concentration of the industry, with no single entity controlling more than 1.0% of revenue
* Decentralized structure of the industry, with thousands of small companies operating locally and focusing on niche market segments
* Geographical diversity, with different regions suited to corn farming and climate variations across provinces creating specific ag

Add the list of answers into our original dataset. Now we have a complete test set that is ready for evaluation.

In [55]:
dataset = dataset.add_column("answer", answers)

In [56]:
type(dataset)

datasets.arrow_dataset.Dataset

## Evaluate the results

#### Preview the final test set

In [57]:
dataset.to_pandas().head(5)

,question,contexts,ground_truth,evolution_type,metadata,episode_done,answer
0,What percentage of structural damage is being ...,"[while choking back tears. \n """"We're seein...",The context reports potentially 30% to 50% str...,simple,"[{'file_name': None, 'page': None, 'source': '...",True,Authorities have estimated that there is poten...
1,What is the expected growth range for adjusted...,[ transportation of various commodities and go...,"According to CN's latest forecast, the expecte...",simple,"[{'file_name': None, 'page': None, 'source': '...",True,I don't know.
2,What is the expected revenue growth for long-d...,[ping t o addr ess l abor short ages. A s the ...,Revenue for long-distance freight trucking ser...,simple,[{'file_name': '/projects/RAG2/scotia-2/Datase...,True,The expected revenue growth for long-distance ...
3,What are the potential economic impacts of a l...,[ binding arbitration is imposed.\n Earlier...,The potential economic impacts of a labor stop...,simple,"[{'file_name': None, 'page': None, 'source': '...",True,A labor stoppage at Canada's two largest railr...
4,What is the purpose of the conciliation proces...,[ions to breakthrough on these paid sick leave...,The purpose of the conciliation process in lab...,simple,"[{'file_name': None, 'page': None, 'source': '...",True,I don't know.


Run the evaluation query to score the results. In this evaluation, we are looking at the following metrics:
- *[Faithfulness](https://docs.ragas.io/en/v0.1.21/concepts/metrics/faithfulness.html)*: Are all the claims that are made in the answer inferred from the given context(s)?
- *[Context Precision](https://docs.ragas.io/en/v0.1.21/concepts/metrics/context_precision.html)*: Did our retriever return good results that matched the question it was being asked?
- *[Answer Correctness](https://docs.ragas.io/en/v0.1.21/concepts/metrics/answer_correctness.html)*: Was the generated answer correct? Was it complete?

In [59]:
%%time
score = evaluate(
    dataset=dataset,
    metrics=[
        Faithfulness(),
        ContextPrecision(),
        AnswerCorrectness(),
    ],
    llm=generator_llm, # Using OpenAI LLM as the evaluator
    embeddings=generator_embeddings,
)


Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]

CPU times: user 19.2 s, sys: 356 ms, total: 19.6 s
Wall time: 2min 4s


In [60]:
%%time
#use OpenAi
# score = evaluate(
#     dataset=dataset,
#     metrics=[
#         Faithfulness(),
#         ContextPrecision(),
#         AnswerCorrectness(),
#     ],
#     llm=generator_llm, # Using OpenAI LLM as the evaluator
#     #llm=llm, # Using Llama LLM as the evaluator
#     embeddings=embeddings,
# )

CPU times: user 9 µs, sys: 1e+03 ns, total: 10 µs
Wall time: 17.2 µs


In [63]:
score.to_pandas().isna().sum(axis =0)

question              0
contexts              0
ground_truth          0
evolution_type        0
metadata              0
episode_done          0
answer                0
faithfulness          0
context_precision     0
answer_correctness    0
dtype: int64

In [67]:
score.to_pandas().tail(4)

,question,contexts,ground_truth,evolution_type,metadata,episode_done,answer,faithfulness,context_precision,answer_correctness
46,What challenges does the Mayan Train project h...,[ by the next five years\nas trade finance oft...,Experts have warned that the Mayan Train has r...,multi_context,"[{'file_name': None, 'page': None, 'source': '...",True,The Mayan Train project has faced significant ...,1.000000,1.0,0.649690
47,What benefits do businesses see from integrate...,[Previous suppl y chain chal lenges ar e pr om...,Businesses see benefits from integrated supply...,multi_context,[{'file_name': '/projects/RAG2/scotia-2/Datase...,True,Businesses see several benefits from integrate...,0.352941,1.0,0.641920
48,What steps did the Bombers take for Indigenous...,[ people from across\nCanada to attend the Win...,"The Bombers, in line with EIC's model, partici...",multi_context,"[{'file_name': None, 'page': None, 'source': '...",True,"The Winnipeg Blue Bombers, in collaboration wi...",0.136364,1.0,0.309660
49,What prompted Mayor Salinas's emergency declar...,[ late 2021 to help fund its\napproximately $3...,Mayor Salinas issued an emergency declaration ...,multi_context,"[{'file_name': None, 'page': None, 'source': '...",True,Mayor Rolando Salinas issued an emergency decl...,1.000000,1.0,0.879351


In [64]:
score["answer_correctness"].mean()

0.5993340782213483

In [65]:
score["faithfulness"].mean()

0.6362367859335891

In [45]:
#score.to_pandas().to_parquet  ("../../Testing_Data/score_llama_answeropenAi.parquet")

In [66]:
score.to_pandas().to_parquet  ("../../Testing_Data/score_llama_answer_openAi_scoreOpenAi_1_DB.parquet")

In [94]:
#"Meta-Llama-3.1-8B-Instruct"
#chunksize, answer correctness, faithfulness, number of question
#10000, 0.592, 0.698, 50
# 5000, 0.617, 0.700, 50
# 3500, 0.677, 0.734, 50
# 3000, 0.688, 0.800, 50
# 2000, 0.629, 0.742, 50
# 1000, 0.644, 0.703, 50
#  500, 0.619, 0.503, 50
#  250, 0.594, 0.527, 50



In [ ]:
#2500, 0.62, 0.825
#3000, 0.60, 0.78 llama, llamaa, llama, llama
#3000, 062, 0.73, llama, llama, openAi, llama